# Original neural computations: explicit APIs, parameters and examples

This notebook exposes the actual computation steps used upstream of Figures 4–7 and S4–S8. Figure notebooks start from the separately released analysis source tables; this notebook requires earlier aligned trial-level inputs as described in `data/README.md`.

Each section defines its notebook-level computation function, its frozen numerical choices and an optional invocation. Shared numerical estimators remain in `src`. All expensive computations are **opt-in**. Outputs go to a new directory; no input arrays or original caches are overwritten. A disabled raw-data section is not evidence that its computation was executed.

For exact stochastic parity, supply the original subject seeds, fixed neuron indices and retained-trial fold assignments. The one-session decoder example is not a substitute for the original full-cohort batch records. Day, mouse, phase and route dimensions remain explicit.

In [ ]:
import os
import sys
from pathlib import Path
from dataclasses import asdict

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "main.py").is_file() and (p / "src/paper.py").is_file())
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
from IPython.display import display

DATA_ROOT = Path(os.environ.get("PPRH_DATA_DIR", str(ROOT / "data/input")))
OUTPUT_ROOT = Path(os.environ.get("PPRH_OUTPUT_DIR", str(ROOT / "outputs/notebooks"))) / "recomputed_metrics"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
F8_MOUSE, F8_GROUP = "A4", "C"
TMAZE_MOUSE, TMAZE_GROUP = "P1", "P"
F8_SUBJECT = DATA_ROOT / "aligned_figure8" / F8_MOUSE
TMAZE_SUBJECT = DATA_ROOT / "aligned_tmaze" / TMAZE_MOUSE

RUN_F8_MANIFOLD = False
RUN_TMAZE_MANIFOLD = False
RUN_STABILITY = False
RUN_F8_PV = False
RUN_TMAZE_PV = False
RUN_DECODER = False


def require_files(subject_dir, names):
    """Check the aligned-data contract before starting an expensive computation."""
    missing = [name for name in names if not (Path(subject_dir) / name).is_file()]
    if missing:
        raise FileNotFoundError(f"Missing aligned inputs: {missing}")


## 1. Figure-8 manifold separation

The per-subject function reads trial maps and labels, keeps the original valid-trial/context rules, samples 50 neurons, embeds in 15 dimensions, computes pooled-covariance separation and standardizes against label shuffles. The repeated-day arrays are kept in the returned cache. The seed must match the original run when exact parity is required.

In [ ]:
from maze_analysis.manifold import compute_manifold_quantification_for_subject, manifold_cache_to_long_rows

F8_MANIFOLD_PARAMETERS = dict(num_day=5, target_neuron_count=50, embedding_dim=15,
                              n_repeat=100, n_shuffle=100, reducer="spectral", zscore=True,
                              min_finite_fraction=.8, dprime_covariance="legacy_pooled")
F8_SUBJECT_SEED = 2026  # Replace with the recorded subject seed for exact stochastic parity.
display(pd.DataFrame([F8_MANIFOLD_PARAMETERS]))


def recompute_figure8_manifold(subject_dir, mouse, group, *, seed):
    """Compute the frozen manifold method while keeping its per-day cache."""
    require_files(subject_dir, ("snarm_1d.npy", "trial_labels.npy"))
    cache = compute_manifold_quantification_for_subject(
        subject_dir, mouse, seed=seed, save_cache=False, **F8_MANIFOLD_PARAMETERS)
    if cache is None:
        raise RuntimeError("No Figure-8 manifold result was returned")
    rows = pd.DataFrame(manifold_cache_to_long_rows(cache, group=group, num_day=5))
    return cache, rows


In [ ]:
if RUN_F8_MANIFOLD:
    cache, rows = recompute_figure8_manifold(F8_SUBJECT, F8_MOUSE, F8_GROUP, seed=F8_SUBJECT_SEED)
    np.save(OUTPUT_ROOT / f"{F8_MOUSE}_manifold_cache.npy", cache, allow_pickle=True)
    rows.to_csv(OUTPUT_ROOT / f"{F8_MOUSE}_manifold_mouse_day_contrast.csv", index=False)
    display(rows.head())


## 2. T-maze manifold separation

The frozen contract retains sample/choice phases, eligible correct trials, route-balanced undersampling and phase-label shuffling separately within left/right routes. `historical_subject_seed` retains the original subject ordering. `cache_to_bin_table` and `bin_to_section_table` preserve day and contrast.

In [ ]:
from maze_analysis.tmaze_legacy_pooled import (
    LegacyPooledConfig, compute_subject_legacy_pooled, historical_subject_seed,
    cache_to_bin_table, bin_to_section_table,
)

TMAZE_MANIFOLD_CONFIG = LegacyPooledConfig(
    days=(1, 2, 3, 4, 5), target_neuron_count=50, embedding_dim=15,
    n_repeat=100, n_shuffle=100, reducer="spectral", zscore=True,
    dprime_covariance="legacy_pooled", min_trials_per_condition=3,
)
display(pd.DataFrame([asdict(TMAZE_MANIFOLD_CONFIG)]))


def recompute_tmaze_manifold(subject_dir, mouse, group):
    """Return the original T-maze cache and explicit bin/section tables."""
    require_files(subject_dir, ("Snarm_trial_v2.npy", "index_all.npy"))
    seed = historical_subject_seed(mouse, TMAZE_MANIFOLD_CONFIG)
    cache = compute_subject_legacy_pooled(subject_dir, mouse=mouse, group=group,
                                         config=TMAZE_MANIFOLD_CONFIG, seed=seed)
    bins = cache_to_bin_table(cache)
    sections = bin_to_section_table(bins)
    return cache, bins, sections


In [ ]:
if RUN_TMAZE_MANIFOLD:
    cache, bins, sections = recompute_tmaze_manifold(TMAZE_SUBJECT, TMAZE_MOUSE, TMAZE_GROUP)
    np.save(OUTPUT_ROOT / f"{TMAZE_MOUSE}_manifold_cache.npy", cache, allow_pickle=True)
    bins.to_csv(OUTPUT_ROOT / f"{TMAZE_MOUSE}_manifold_bins.csv", index=False)
    sections.to_csv(OUTPUT_ROOT / f"{TMAZE_MOUSE}_manifold_sections.csv", index=False)
    display(sections.head())


## 3. Figure-8 odd/even and interday stability

The dictionaries below are copied from the accepted historical `Fig4_run_manifest.json` analysis configurations 14, 16 and 17 (current manuscript Fig.5). Odd/even splitting is condition-ordinal; the minimum four trials supports both halves. Single-neuron correlations are Pearson, PV correlations are Spearman, and adjacent/D1-anchor pair modes remain separate.

In [ ]:
from maze_analysis.pv_pc_subjects import (
    compute_pv_within_reliability_subject, compute_intraday_pc_within_subject,
    compute_interday_stability_subject,
)

WITHIN_PV_CONFIG = {'distance_threshold': 6.0, 'min_split_trials': 4, 'n_subsample_iter': 100, 'num_day': 5, 'pv_method': 'spearman', 'pv_neuron_mode': 'all50', 'pv_pair': 'CL_CR', 'smooth': True, 'split_strategy': 'condition_ordinal_odd_even_v1', 'transition_penalty': 0.25}
WITHIN_NEURON_CONFIG = {'condition_keys': ['CL', 'CR'], 'distance_threshold': 6.0, 'min_split_trials': 4, 'num_day': 5, 'pc_method': 'pearson', 'smooth': True, 'split_strategy': 'condition_ordinal_odd_even_v1', 'transition_penalty': 0.25}
INTERDAY_CONFIG = {'n_subsample_iter': 100, 'num_day': 5, 'pair_modes': ['adjacent', 'anchor'], 'pc_method': 'pearson', 'pv_method': 'spearman', 'pv_neuron_mode': 'all50', 'pv_pair': 'CL_CR', 'random_seed': 2026}
display(pd.DataFrame([WITHIN_PV_CONFIG, WITHIN_NEURON_CONFIG, INTERDAY_CONFIG],
                     index=["within_PV", "within_neuron", "interday"]))


def recompute_figure8_stability(subject_dir, mouse, group):
    """Keep within-day reliability and both interday pair modes as separate outputs."""
    require_files(subject_dir, ("xy_trial_all.npy", "d_dff_trial_all.npy", "trial_labels.npy", "snarm_all.npy"))
    within_pv = compute_pv_within_reliability_subject(subject_dir, mouse, group, WITHIN_PV_CONFIG)
    within_neuron = compute_intraday_pc_within_subject(subject_dir, mouse, group, WITHIN_NEURON_CONFIG)
    interday = compute_interday_stability_subject(subject_dir, mouse, group, INTERDAY_CONFIG)
    return dict(within_pv=within_pv, within_neuron=within_neuron, interday=interday)


In [ ]:
if RUN_STABILITY:
    stability = recompute_figure8_stability(F8_SUBJECT, F8_MOUSE, F8_GROUP)
    np.save(OUTPUT_ROOT / f"{F8_MOUSE}_stability.npy", stability, allow_pickle=True)
    stability["within_neuron"]["long_table"].to_csv(OUTPUT_ROOT / f"{F8_MOUSE}_within_neuron.csv", index=False)
    for pair_mode, rows in stability["interday"]["pc_long"].items():
        rows.to_csv(OUTPUT_ROOT / f"{F8_MOUSE}_interday_{pair_mode}_neuron.csv", index=False)


## 4. Figure-8 PV shuffle separation

The original function computes Spearman PV correlation and condition-label shuffle distributions before reporting positive separation as `(mean_shuffle - observed) / sd_shuffle`. Subject/group indices affect historical random seeds; fill them from the original run record rather than reindexing a reduced cohort.

In [ ]:
from maze_analysis.pv_shuffle import compute_pv_shuffle_control_for_subject as compute_figure8_pv
from maze_analysis.pv_shuffle import resolve_pv_pair_config

F8_GROUP_INDEX = None
F8_MOUSE_INDEX = None


def recompute_figure8_pv(subject_dir, mouse, *, group_index, mouse_index):
    """Use all-neuron fixed-50 sampling and the original shuffle-control function."""
    require_files(subject_dir, ("snarm_all.npy", "xy_trial_all.npy", "d_dff_trial_all.npy", "trial_labels.npy"))
    if group_index is None or mouse_index is None:
        raise ValueError("Supply the original group and mouse indices used by the frozen seed schedule")
    pair = resolve_pv_pair_config("CL_CR", "all50")
    cache = compute_figure8_pv(
        str(subject_dir), mouse, int(group_index), int(mouse_index), num_day=5,
        selected_trials=pair["selected_trials"], selected_control=True,
        pc_target_by_day=np.full(5, 50), n_subsample_iter=100,
        n_condition_shuffle_iter=100, n_within_condition_split_iter=100,
        fixed_pc_target=50, pv_neuron_mode="all50",
    )
    if cache is None:
        raise RuntimeError("No Figure-8 PV cache was returned")
    return cache


In [ ]:
if RUN_F8_PV:
    cache = recompute_figure8_pv(F8_SUBJECT, F8_MOUSE, group_index=F8_GROUP_INDEX, mouse_index=F8_MOUSE_INDEX)
    np.save(OUTPUT_ROOT / f"{F8_MOUSE}_pv_shuffle.npy", cache, allow_pickle=True)


## 5. T-maze PV separation

Sample left/right, choice left/right and sample/choice comparisons remain distinct. The sample/choice contrast uses route-balanced sampling and phase shuffles within route. Supply the recorded PV seed before enabling the call. The figure notebooks plot only the Spearman separation metric.

In [ ]:
from t_maze.pv_shuffle import compute_pv_shuffle_control_for_subject as compute_tmaze_pv

TMAZE_PV_SEED = None


def recompute_tmaze_pv(subject_dir, mouse, group, *, seed):
    """Compute fixed-50 Spearman separation with day and contrast retained."""
    require_files(subject_dir, ("Snarm_trial_v2.npy", "index_all.npy"))
    if seed is None:
        raise ValueError("Supply the PV subject seed recorded by the original run")
    cache = compute_tmaze_pv(subject_dir, mouse_name=mouse, group=group, num_day=5,
                             target_neuron_count=50, n_repeat=100, n_shuffle=100,
                             min_trials_per_condition=3, seed=int(seed), zscore=True,
                             save_cache=False, metrics=("spearman",))
    if cache is None:
        raise RuntimeError("No T-maze PV cache was returned")
    return cache


In [ ]:
if RUN_TMAZE_PV:
    cache = recompute_tmaze_pv(TMAZE_SUBJECT, TMAZE_MOUSE, TMAZE_GROUP, seed=TMAZE_PV_SEED)
    np.save(OUTPUT_ROOT / f"{TMAZE_MOUSE}_pv_shuffle.npy", cache, allow_pickle=True)


## 6. One-session Gaussian Bayesian decoder and within-trial shifts

The function below shows the actual train/predict/shift sequence. Supply the original aligned position/neural trial arrays, labels, fixed 50-neuron indices and **retained-trial** fold assignments (five folds). Model moments exclude held-out trials. Circular shifts occur within each trial and retain its route identity; this null is not a route-information test.

The example returns held-out predictions, a soft posterior matrix, path-error metrics and 100 shift-control summaries. It does not reconstruct the entire original cohort batch or assert exact reproduction of the published Euclidean-error table; Figure S4/S7 read that released table explicitly.

In [ ]:
from maze_analysis.bayesian_position import prepare_binned_trials, bin_geometry, shift_bins_within_trials, decoding_metrics
from maze_analysis.bayesian_position_batch import TrialGaussianCV
from maze_analysis.position_posterior import mean_posterior_matrix

DECODER_INPUTS = None  # Mapping with the keyword arrays required by decode_one_session below.


def decode_one_session(xy_trials, neural_trials, labels, neuron_indices, folds, shuffle_seed, context=None):
    """Run the shared five-trial-fold decoder with the supplied frozen session choices."""
    neuron_indices = np.asarray(neuron_indices, dtype=int)
    if len(neuron_indices) != 50 or len(np.unique(neuron_indices)) != 50:
        raise ValueError("The manuscript session uses 50 unique fixed neuron indices")
    trials, qc = prepare_binned_trials(xy_trials, neural_trials, labels, neuron_indices, context=context)
    folds = np.asarray(folds, dtype=int)
    if folds.shape != (len(trials),) or not np.array_equal(np.unique(folds), np.arange(5)):
        raise ValueError("Supply five-fold assignments for the retained trial list")
    cv = TrialGaussianCV(trials, folds=folds)
    prediction, posterior, training_counts = cv.predict(return_posterior=True)
    mean_posterior, true_bin_counts = mean_posterior_matrix(cv.y, posterior)
    _, path_distances = bin_geometry()
    observed = decoding_metrics(cv.y, prediction, path_distances)
    rng = np.random.default_rng(int(shuffle_seed))
    shift_rows = []
    for repeat in range(100):
        shifted, offsets = shift_bins_within_trials(trials, rng)
        shifted_prediction, _, _ = cv.predict(np.concatenate(shifted))
        metrics = decoding_metrics(cv.y, shifted_prediction, path_distances)
        shift_rows.append(dict(repeat=repeat, **metrics))
    return dict(prediction=prediction, mean_posterior=mean_posterior,
                true_bin_counts=true_bin_counts, training_counts=training_counts,
                observed_path_metrics=observed, shift_path_metrics=pd.DataFrame(shift_rows),
                trial_qc=pd.DataFrame(qc))


In [ ]:
if RUN_DECODER:
    if DECODER_INPUTS is None:
        raise ValueError("Populate DECODER_INPUTS with the original session arrays and frozen choices")
    result = decode_one_session(**DECODER_INPUTS)
    np.savez_compressed(OUTPUT_ROOT / "decoder_session_outputs.npz",
                        prediction=result["prediction"], mean_posterior=result["mean_posterior"],
                        true_bin_counts=result["true_bin_counts"], training_counts=result["training_counts"])
    result["shift_path_metrics"].to_csv(OUTPUT_ROOT / "decoder_session_shift_path_metrics.csv", index=False)
    result["trial_qc"].to_csv(OUTPUT_ROOT / "decoder_session_trial_qc.csv", index=False)
    display(pd.DataFrame([result["observed_path_metrics"]]))


## 7. From neural outputs to manuscript figures

The computation functions above keep original day/contrast arrays or mouse-day tables. They do not replace the released analysis source data automatically. To reproduce accepted panels and their longitudinal inference, use the current figure notebooks with the exact source-data inventory. Full-cohort raw reruns and their stochastic parity require the original aligned-data/session records and separate validation. No raw run is claimed when the opt-in switches above remain disabled.